# Phase 4 - P4-E04: SAR Temporal Flood Inundation Validation

## Primary Benchmark Provenance & Specification
- **Dataset**: Modified Sen1Floods11 Dataset for Change Detection
- **DOI**: [10.5281/zenodo.7946594](https://doi.org/10.5281/zenodo.7946594) (Concept DOI: 10.5281/zenodo.7946593)
- **Version**: v1 (Publication date: 2023-05-17)
- **Creator**: Ritu Yadav / KTH Royal Institute of Technology
- **License**: Creative Commons Attribution 4.0 International (CC BY 4.0)
- **Associated Paper**: "Attentive Dual Stream Siamese U-Net for Flood Detection on Multi-Temporal Sentinel-1 Data", IGARSS 2022 (DOI: [10.1109/IGARSS46834.2022.9883132](https://doi.org/10.1109/IGARSS46834.2022.9883132))

### Pinned Authoritative Files & Published Hashes
1. `PRE_S1-20230517T191707Z-001.zip` (1,520,699,949 bytes, MD5: `4a32637c56ea519bd3c4baca208b289d`)
2. `POST_S1-20230517T191716Z-001.zip` (729,719,788 bytes, MD5: `40a505cfbd5318d94a9d5bd7aef88561`)
3. `Labels-20230517T191741Z-001.zip` (2,462,219 bytes, MD5: `069b4c05eefb7a6e72c1adb34aaf1a24`)

### Label Semantic Audit & Evaluation Target
- **Audit Finding**: In Modified Sen1Floods11, labels delineate **post-event water/flood extent** (1 = water/inundated, 0 = non-water, -1 = NoData). They do **NOT** pre-subtract permanent baseline water.
- **Evaluation Target**: Model performance is evaluated against the authoritative post-event water extent ground truth.
- **SatQuery Deterministic Evidence**: SatQuery computes the bi-temporal flood expansion / newly inundated change (post-water minus pre-water, backscatter drop >= 3 dB) as a separate deterministic GIS evidence computation (`flood_expansion_m2`).
- **Benchmark Policy**: Original single-date Sen1Floods11 is NOT used alone as temporal change proof. No random mirrors are permitted.

In [ ]:
# Install dependencies
!pip install -q numpy rasterio affine

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

print('Initializing Phase 4 E04 SAR Temporal/Flood Evaluation...')

assert 'SATQUERY_REMOTE_OUTPUT' in os.environ, 'SATQUERY_REMOTE_OUTPUT environment variable missing'
output_dir = Path('/kaggle/working/satquery-output') / os.environ['SATQUERY_REMOTE_OUTPUT']
output_dir.mkdir(parents=True, exist_ok=True)

# Add satquery repo to path
repo_root = Path('/kaggle/working/SATQuery')
if not repo_root.exists():
    repo_url = os.environ.get('SATQUERY_REPO_URL', 'https://github.com/bishuk-dev/SIH-26167-SATQuery.git')
    print('Cloning ' + repo_url + '...')
    subprocess.run(['git', 'clone', repo_url, str(repo_root)], check=True)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
from scripts.kaggle.p4_e04_baseline import run_p4_e04_evaluation

metrics = run_p4_e04_evaluation(output_dir)
status = metrics.get('status')
print('P4-E04 evaluation completed with status:', status)

if status == 'PASS':
    expected_files = ['sar_validation_metrics.json', 'sar_validation_predictions.jsonl', 'runner_meta.json', 'evaluation_meta.json']
    for fname in expected_files:
        fpath = output_dir / fname
        if not fpath.exists():
            raise FileNotFoundError('Expected result file missing after evaluation: ' + str(fpath))
        print('  Verified: ' + fpath.name + ' (' + str(fpath.stat().st_size) + ' bytes)')
    print('Evaluation complete!')
else:
    print('Evaluation did not pass. Check evaluation_failure.json or sar_validation_metrics.json for details.')